# Music DNA — Interactive Audio & Research Exploration Notebook

This companion notebook demonstrates the complete end-to-end Music DNA audio intelligence framework on real-world recordings, featuring **Amir Khusrau's classical Sufi masterpiece *Chaap Tilak Sab Cheeni* (Abida Parveen & Rahat Fateh Ali Khan)**.

### Research Pipeline Explored
1. **Audio Playback & Waveform Inspection**: Inline HTML5 playback and time-domain analysis
2. **Deterministic DSP Acoustic Descriptors**: Physical descriptors via `AcousticFeatureExtractor`
3. **Deep Neural Stem Separation**: Meta Demucs (`htdemucs`) 6-stem decomposition (`vocals`, `drums`, `bass`, `other`, `guitar`, `piano`)
4. **Musical Dimensional Analysis**: Key (Krumhansl-Schmuckler), chords, tempo & syncopation, and instrument probabilities
5. **Deep Neural Encoding & Multimodal Fusion**: 64-D spectro-temporal transformer fused with 25-D DSP into 89-D space
6. **Multi-Dimensional Latent Space Geometry**: 3-D PCA spatial trajectory, scree variance curve, and pairwise latent projections
7. **Continuous Dynamic Emotion Trajectories**: AAAI 2025 dual-scale attention and time-varying Valence/Arousal affect curves
8. **Active Inquiry & Information-Gain Optimization**: Shannon entropy question formulation and query-vs-random simulation
9. **FAV Corpus (15s Favorite Passages)**: Explanation taxonomy and passage-vs-track acoustic contrast corroboration
10. **Complete Research Benchmark Suite**: End-to-end multi-pillar scientific evaluation
11. **Unified Multi-Modal Time-Aligned Dashboard**: Synchronized time-domain waveform, high-resolution STFT spectrogram, Demucs 4-stem lanes, physical DSP trajectories, continuous affect curves, and latent principal component trajectories on a shared X-axis.


In [ ]:
# Setup autoreload and interactive environment
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt

# Enable high-DPI Retina display mode for crystal-clear inline plots
try:
    get_ipython().run_line_magic("config", "InlineBackend.figure_format = 'retina'")
except Exception:
    pass

plt.rcParams["figure.dpi"] = 130
plt.rcParams["savefig.dpi"] = 200

# Auto-install audio dependencies into whatever kernel is currently selected
try:
    import librosa
    import soundfile
    import sklearn
    import matplotlib
    import torch
    import demucs
except ImportError:
    print("Installing audio & deep learning dependencies into active Python environment...")
    !{sys.executable} -m pip install -q librosa soundfile scikit-learn matplotlib demucs torch torchaudio

# Robust repository root discovery across local IDEs, Jupyter Lab, and Google Colab
def resolve_workspace_root() -> Path:
    candidates = []
    curr = Path.cwd().resolve()
    for _ in range(6):
        candidates.append(curr)
        if curr.parent == curr:
            break
        curr = curr.parent

    content_dir = Path("/content")
    if content_dir.is_dir():
        for sub in content_dir.iterdir():
            if sub.is_dir():
                candidates.append(sub)
        candidates.append(content_dir)

    for candidate in candidates:
        if (candidate / "services" / "intelligence").is_dir():
            return candidate.resolve()

    return Path.cwd().resolve()

workspace_root = resolve_workspace_root()

if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))
os.chdir(str(workspace_root))

if not (workspace_root / "services" / "intelligence").is_dir():
    err_msg = "Could not locate services/intelligence. In Colab run: !git clone <repo> && %cd <repo>"
    raise ModuleNotFoundError(err_msg)

import math
import numpy as np
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
import soundfile as sf
from IPython.display import Audio, display

from services.intelligence.acoustic_features import AcousticFeatureExtractor, SAMPLE_RATE, analyze_track
from services.intelligence.deep_encoder import DeepAudioEncoder, MultimodalFusionEngine, analyze_track_deep, fuse_multimodal_embeddings
from services.intelligence.stem_separation import StemSeparator, MusicalStructureAnalyzer, separate_stems, analyze_musical_dimensions
from services.intelligence.vector_projection import EmbeddingProjector, project_to_2d, project_multidimensional
from research.active_learning import ActiveLearningEngine, select_inquiry_candidate, formulate_inquiry_prompt
from research.dynamic_emotion import DynamicEmotionModel, extract_sequence_features, compute_dual_scale_attention
from research.favorite_passages import FavoritePassageTaxonomy, AcousticContrastAnalyzer, classify_fav_explanation, calculate_acoustic_contrast
from research.benchmark_suite import ResearchBenchmarkSuite, BenchmarkSuite

# Primary subject: Complete original MP3 recording
demo_audio_path = workspace_root / "public" / "Chaap-Tilak-Sab-Cheeni-Abida-Parveen-Rahat-Fateh-Ali.mp3"

print(f"✓ Environment ready. Active Python: {sys.executable}")
print(f"✓ Workspace root: {workspace_root}")
print(f"✓ Active subject audio: {demo_audio_path.name} (exists: {demo_audio_path.is_file()})")


## 1. Subject Playback & Waveform Inspection
Listen to the recording with an inline player and inspect the time-domain waveform.

In [ ]:
# Configure start offset and selected duration to analyze
# Selected duration: 3 minutes (180.0 seconds) of the complete MP3 recording
start_offset = 0.0
playback_duration = 180.0

audio_waveform, sample_rate = librosa.load(
    str(demo_audio_path), sr=SAMPLE_RATE, mono=True, offset=start_offset, duration=playback_duration
)
duration_seconds = len(audio_waveform) / sample_rate
print(f"Audio file: {demo_audio_path.name}")
print(f"Loaded span: {start_offset:.1f}s - {start_offset + duration_seconds:.1f}s ({duration_seconds:.2f}s duration, {sample_rate} Hz)")

# Interactive HTML5 audio player widget
display(Audio(audio_waveform, rate=sample_rate))

# Plot waveform
plt.figure(figsize=(12, 3))
time_axis = np.linspace(start_offset, start_offset + duration_seconds, len(audio_waveform))
plt.plot(time_axis, audio_waveform, color="#8EC5B8", alpha=0.85, linewidth=0.8)
plt.title(f"Audio Waveform (Time Domain) — {demo_audio_path.name} (3 min)")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.xlim(start_offset, start_offset + duration_seconds)
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


## 2. Deterministic DSP Acoustic Analysis
Extract time-aligned windows using the object-oriented  to inspect physical descriptors over time.

In [ ]:
extractor = AcousticFeatureExtractor()
track_analysis = extractor.extract_from_waveform(audio_waveform, sample_rate, start_offset=start_offset)
segments = track_analysis["segments"]
print(f"Extracted {len(segments)} windows ({track_analysis['windowSec']}s windows, {track_analysis['hopSec']}s hop).")

midpoints = [(s["startSec"] + s["endSec"]) / 2.0 for s in segments]
energies = [s["descriptors"]["energy"] for s in segments]
centroids = [s["descriptors"]["brightness_hz"] for s in segments]
onsets = [s["descriptors"]["onset_activity"] for s in segments]
vocals = [s["descriptors"]["vocal_presence"] for s in segments]

fig, axes = plt.subplots(2, 2, figsize=(13, 6), sharex=True)
axes[0, 0].plot(midpoints, energies, marker="o", color="#E06C75")
axes[0, 0].set_title("Energy (RMS)")
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(midpoints, centroids, marker="o", color="#D19A66")
axes[0, 1].set_title("Brightness / Spectral Centroid (Hz)")
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(midpoints, onsets, marker="o", color="#98C379")
axes[1, 0].set_title("Onset Activity / Rhythm")
axes[1, 0].set_xlabel("Time (s)")
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(midpoints, vocals, marker="o", color="#61AFEF")
axes[1, 1].set_title("Vocal Presence Ratio")
axes[1, 1].set_xlabel("Time (s)")
axes[1, 1].grid(True, alpha=0.3)

axes[0, 0].set_xlim(start_offset, start_offset + duration_seconds)
plt.tight_layout()
plt.show()


## 3. Deep Neural Stem Separation (Meta Demucs v4)
Decompose audio into isolated stems (`vocals`, `drums`, `bass`, `other`, `guitar`, `piano`) using Meta's **Demucs v4 Hybrid Transformer (`htdemucs_6s`)**.

In [ ]:
stem_separator = StemSeparator()
# Dynamic stem cache indexed by filename, offset, and duration to eliminate cross-file contamination
stem_cache_dir = workspace_root / "services" / "intelligence" / "cache" / "stems" / f"{demo_audio_path.stem}_{int(start_offset)}_{int(duration_seconds)}s"
demucs_stems = {}

all_possible_stems = ["vocals", "drums", "bass", "other", "guitar", "piano"]
use_cache = False
if stem_cache_dir.is_dir():
    existing_stems = [s for s in all_possible_stems if (stem_cache_dir / f"{s}.wav").is_file()]
    if "vocals" in existing_stems and (len(existing_stems) >= 4):
        cached_info = sf.info(str(stem_cache_dir / "vocals.wav"))
        if abs(cached_info.duration - duration_seconds) < 0.2:
            use_cache = True

if use_cache:
    cached_stems = [s for s in all_possible_stems if (stem_cache_dir / f"{s}.wav").is_file()]
    stems_str = ', '.join(cached_stems)
    print(f"Loading cached Meta Demucs {len(cached_stems)}-stem separations ({stems_str}) from: {stem_cache_dir.name}")
    for s in cached_stems:
        stem_y, _ = librosa.load(str(stem_cache_dir / f"{s}.wav"), sr=sample_rate, mono=True)
        demucs_stems[s] = stem_y[:len(audio_waveform)]
else:
    print(f"Running Meta Demucs (htdemucs_6s) Deep Neural Stem Separation ({duration_seconds:.1f}s audio)...")
    demucs_stems = stem_separator.separate(audio_waveform, sample_rate, model_family="demucs_6s")
    stem_cache_dir.mkdir(parents=True, exist_ok=True)
    for s, data in demucs_stems.items():
        sf.write(str(stem_cache_dir / f"{s}.wav"), data, sample_rate)

print()
print(f"--- Interactive Demucs Deep Stem Audio Playback ({len(demucs_stems)} Stems) ---")
for stem_name, stem_audio in demucs_stems.items():
    stem_rms = float(np.sqrt(np.mean(stem_audio**2)))
    print(f"Stem: {stem_name.upper():<7} (RMS: {stem_rms:.4f})")
    display(Audio(stem_audio, rate=sample_rate))

# Plot isolated stem waveforms
n_stems = len(demucs_stems)
fig, axes = plt.subplots(n_stems, 1, figsize=(12, 1.6 * n_stems), sharex=True)
if n_stems == 1:
    axes = [axes]
stem_colors = {
    "vocals": "#E06C75",
    "other": "#61AFEF",
    "drums": "#98C379",
    "bass": "#D19A66",
    "guitar": "#E5C07B",
    "piano": "#C678DD",
}
t_axis = np.linspace(start_offset, start_offset + duration_seconds, len(audio_waveform))
for ax, (stem_name, stem_audio) in zip(axes, demucs_stems.items()):
    ax.plot(t_axis, stem_audio, color=stem_colors.get(stem_name, "#ABB2BF"), alpha=0.85, linewidth=0.7)
    ax.set_ylabel(stem_name.upper(), fontsize=9, fontweight="bold")
    ax.grid(True, alpha=0.25)
axes[-1].set_xlabel("Time (seconds)")
axes[-1].set_xlim(start_offset, start_offset + duration_seconds)
plt.suptitle(f"Demucs Deep Neural {n_stems}-Stem Waveforms — {demo_audio_path.name}", fontsize=12, y=0.99)
plt.tight_layout()
plt.show()


## 4. Musical Dimensional Structure Analysis
Extract the five fundamental musical dimensions from the isolated Demucs stems: Key, Chord Progression, Tempo, Syncopation, Probabilistic Instruments, and Arrangement Dynamics.

In [ ]:
structure_analyzer = MusicalStructureAnalyzer()
dimensions = structure_analyzer.analyze_dimensions(audio_waveform, sample_rate, stems=demucs_stems)

print("=== Musical Dimensions Analysis (Derived from Demucs Stems) ===")
print(f"1. Melody & Harmony : Key = {dimensions['melodyHarmony']['key']} (Confidence: {dimensions['melodyHarmony']['keyConfidence']:.2f})")
print(f"   Chord Progression : {dimensions['melodyHarmony']['chords']}")
print(f"   Harmonic Stability: {dimensions['melodyHarmony']['harmonicStability']:.3f}")
print(f"2. Rhythm & Groove  : Tempo = {dimensions['rhythmGroove']['tempoBpm']:.1f} BPM (Syncopation: {dimensions['rhythmGroove']['syncopation']:.3f})")
print(f"3. Instrumentation  : Top = {dimensions['instrumentationTimbre']['topInstruments']}, Texture = {dimensions['instrumentationTimbre']['texture']}")
print(f"4. Vocals           : Presence = {dimensions['vocalsExpression']['presence']:.3f}, Breathiness = {dimensions['vocalsExpression']['breathiness']:.3f}")
print(f"5. Arrangement      : Active Stems = {dimensions['arrangementTransitions']['activeStems']}, Transition = {dimensions['arrangementTransitions']['transitionType']}")


## 5. Deep Neural Encoding & Multimodal Fusion
Compute 64-D neural embeddings from the spectro-temporal transformer and fuse with 25-D physical DSP descriptors into a unified 89-D unit-normalized representation.

In [ ]:
deep_encoder = DeepAudioEncoder()
deep_analysis = deep_encoder.analyze_waveform_deep(
    audio_waveform=audio_waveform,
    sample_rate=sample_rate,
    base_analysis=track_analysis,
    start_offset=start_offset
)
fused_analysis = MultimodalFusionEngine.fuse(track_analysis, deep_analysis, deep_weight=0.7)

dsp_dim = len(track_analysis["segments"][0]["embedding"])
deep_dim = len(deep_analysis["segments"][0]["embedding"])
fused_dim = len(fused_analysis["segments"][0]["embedding"])

print(f"Physical DSP Dimension    : {dsp_dim}D")
print(f"Deep Transformer Dimension: {deep_dim}D (Model: {deep_analysis['modelId']})")
print(f"Fused Multimodal Dimension: {fused_dim}D (w_deep=0.7, w_dsp=0.3)")

# Verify unit-norm preservation
for seg in fused_analysis["segments"]:
    norm = np.linalg.norm(seg["embedding"])
    assert math.isclose(norm, 1.0, rel_tol=1e-4)
print(f"✓ Unit-norm preservation mathematically verified across all {len(fused_analysis['segments'])} fused segments.")


## 6. Multi-Dimensional Latent Space & Manifold Geometry
Instead of collapsing exclusively to 2D, we project the 89-dimensional multimodal latent space into **3-D spatial trajectories**, analyze the **Scree variance spectrum**, and visualize **multi-dimensional pairwise subspaces**.

In [ ]:
embeddings = [s["embedding"] for s in fused_analysis["segments"]]
projector = EmbeddingProjector()

# Dynamic PCA dimensioning with safe handling of any number of segments
n_comp = min(3, len(embeddings))
if len(embeddings) >= 3:
    coords_raw = np.array(projector.project_multidimensional(embeddings, n_components=n_comp))
else:
    coords_raw = np.ones((len(embeddings), 3)) * 0.5

if coords_raw.shape[1] < 3:
    coords_3d = np.pad(coords_raw, ((0, 0), (0, 3 - coords_raw.shape[1])), mode="constant", constant_values=0.5)
else:
    coords_3d = coords_raw

# Scree variance curve
variance_ratios = projector.get_explained_variance_ratio(embeddings, n_components=min(8, len(embeddings)))
cumulative_variance = np.cumsum(variance_ratios)

fig = plt.figure(figsize=(14, 6))

# Subplot 1: 3-D Spatial Trajectory
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
scatter = ax1.scatter(
    coords_3d[:, 0], coords_3d[:, 1], coords_3d[:, 2],
    c=midpoints, cmap="viridis", s=90, edgecolors="black", alpha=0.85
)
ax1.plot(coords_3d[:, 0], coords_3d[:, 1], coords_3d[:, 2], color="gray", linestyle="--", alpha=0.5)
ax1.set_title("3-D Multimodal Acoustic Latent Space (PCA)")
ax1.set_xlabel("PC 1")
ax1.set_ylabel("PC 2")
ax1.set_zlabel("PC 3")
fig.colorbar(scatter, ax=ax1, shrink=0.6, label="Song Timeline (s)")

# Subplot 2: Multi-Dimensional Scree Plot & Cumulative Variance
ax2 = fig.add_subplot(1, 2, 2)
pc_indices = range(1, len(variance_ratios) + 1)
ax2.bar(pc_indices, variance_ratios, color="#61AFEF", alpha=0.75, label="Individual Explained Variance")
ax2.step(pc_indices, cumulative_variance, where="mid", color="#E06C75", linewidth=2, label="Cumulative Explained Variance")
ax2.set_title("Multi-Dimensional Latent Manifold Scree Plot")
ax2.set_xlabel("Principal Component Index")
ax2.set_ylabel("Explained Variance Ratio")
ax2.set_ylim(0, 1.05)
ax2.grid(True, alpha=0.3)
ax2.legend(loc="lower right")

plt.tight_layout()
plt.show()


## 7. Dynamic Emotion Trajectories & Russell's 2D Circumplex Phase Space (AAAI 2025)
Extract sequence features and apply dual-scale temporal attention to predict continuous Valence and Arousal affective trajectories, and project the musical journey into Russell's 2D Circumplex Phase Space across 4 affective quadrants (Exuberant, Turbulent, Melancholic, Calm).

In [ ]:
emotion_model = DynamicEmotionModel()
raw_seq = emotion_model._extract_sequence_features(fused_analysis["segments"])
attended_seq = emotion_model.compute_dual_scale_attention(raw_seq)

valence = np.tanh((attended_seq[:, 5] - np.mean(attended_seq[:, 5])) * 3.0)
arousal = np.tanh((attended_seq[:, 0] - np.mean(attended_seq[:, 0])) * 2.5 + attended_seq[:, 4] * 1.5)

# --- Dual Visualization: Dynamic Affect Curves + Russell 2D Circumplex Phase Space ---
fig, (ax_traj, ax_circ) = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={"width_ratios": [1.2, 1.0]})

# Panel 1: Time-domain continuous trajectories
ax_traj.plot(midpoints, valence, label="Valence (Positivity)", color="#98C379", marker="s", markersize=4, linewidth=1.8)
ax_traj.plot(midpoints, arousal, label="Arousal (Energy / Tension)", color="#E06C75", marker="^", markersize=4, linewidth=1.8)
ax_traj.axhline(0.0, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
ax_traj.fill_between(midpoints, 0, valence, where=(valence >= 0), color="#98C379", alpha=0.15)
ax_traj.fill_between(midpoints, 0, valence, where=(valence < 0), color="#98C379", alpha=0.08)
ax_traj.fill_between(midpoints, 0, arousal, where=(arousal >= 0), color="#E06C75", alpha=0.15)
ax_traj.fill_between(midpoints, 0, arousal, where=(arousal < 0), color="#E06C75", alpha=0.08)
ax_traj.set_title("Continuous Dynamic Affect Trajectories (AAAI 2025)", fontsize=11, fontweight="bold")
ax_traj.set_xlabel("Song Time (seconds)", fontsize=10)
ax_traj.set_ylabel("Normalized Affect [-1, +1]", fontsize=10)
ax_traj.set_xlim(start_offset, start_offset + duration_seconds)
ax_traj.set_ylim(-1.05, 1.05)
ax_traj.grid(True, alpha=0.3)
ax_traj.legend(loc="upper right", framealpha=0.9, fontsize=9)

# Panel 2: Russell's 2D Circumplex Phase Space
ax_circ.set_facecolor("#1E222A")
ax_circ.axhline(0.0, color="#5C6370", linestyle="--", linewidth=1.0, alpha=0.8)
ax_circ.axvline(0.0, color="#5C6370", linestyle="--", linewidth=1.0, alpha=0.8)

# Quadrant Background Color Shading
ax_circ.fill_between([0, 1.05], 0, 1.05, color="#98C379", alpha=0.06)  # Q1 Exuberant
ax_circ.fill_between([-1.05, 0], 0, 1.05, color="#E06C75", alpha=0.06) # Q2 Turbulent
ax_circ.fill_between([-1.05, 0], -1.05, 0, color="#61AFEF", alpha=0.06) # Q3 Melancholic
ax_circ.fill_between([0, 1.05], -1.05, 0, color="#E5C07B", alpha=0.06)  # Q4 Calm

# Quadrant Emotion Labels
ax_circ.text(0.55, 0.85, "EXUBERANT\n(Joyful / Climactic)", color="#98C379", fontsize=8.5, fontweight="bold", ha="center", va="center", bbox=dict(boxstyle="round,pad=0.3", facecolor="#282C34", edgecolor="#98C379", alpha=0.7))
ax_circ.text(-0.55, 0.85, "TURBULENT\n(Tense / Dramatic)", color="#E06C75", fontsize=8.5, fontweight="bold", ha="center", va="center", bbox=dict(boxstyle="round,pad=0.3", facecolor="#282C34", edgecolor="#E06C75", alpha=0.7))
ax_circ.text(-0.55, -0.85, "MELANCHOLIC\n(Somber / Reflective)", color="#61AFEF", fontsize=8.5, fontweight="bold", ha="center", va="center", bbox=dict(boxstyle="round,pad=0.3", facecolor="#282C34", edgecolor="#61AFEF", alpha=0.7))
ax_circ.text(0.55, -0.85, "CALM\n(Serene / Peaceful)", color="#E5C07B", fontsize=8.5, fontweight="bold", ha="center", va="center", bbox=dict(boxstyle="round,pad=0.3", facecolor="#282C34", edgecolor="#E5C07B", alpha=0.7))

# Trajectory path connecting consecutive time windows
ax_circ.plot(valence, arousal, color="#ABB2BF", linestyle="-", linewidth=1.2, alpha=0.6, zorder=2)

# Scatter points colored by timeline progression
scatter = ax_circ.scatter(valence, arousal, c=midpoints, cmap="plasma", s=45, edgecolor="#FFFFFF", linewidth=0.6, zorder=4)
cbar_circ = plt.colorbar(scatter, ax=ax_circ, pad=0.03, fraction=0.046)
cbar_circ.set_label("Timeline (s)", fontsize=9)

# Highlight Start (t=0) and End (t=T)
ax_circ.scatter(valence[0], arousal[0], s=120, color="#50A14F", edgecolor="#FFFFFF", linewidth=1.5, marker="D", zorder=5, label=f"Start ({midpoints[0]:.1f}s)")
ax_circ.scatter(valence[-1], arousal[-1], s=130, color="#E45649", edgecolor="#FFFFFF", linewidth=1.5, marker="*", zorder=5, label=f"End ({midpoints[-1]:.1f}s)")

ax_circ.set_xlim(-1.05, 1.05)
ax_circ.set_ylim(-1.05, 1.05)
ax_circ.set_xlabel("Valence (Negative ← → Positive)", fontsize=10, fontweight="bold")
ax_circ.set_ylabel("Arousal (Calm ← → Excited)", fontsize=10, fontweight="bold")
ax_circ.set_title("Russell's 2D Circumplex Phase Space", fontsize=11, fontweight="bold")
ax_circ.legend(loc="lower left", fontsize=8, framealpha=0.85)

plt.tight_layout()
plt.show()

# Circumplex Quadrant Distribution Statistics
q1_count = int(np.sum((valence >= 0) & (arousal >= 0)))
q2_count = int(np.sum((valence < 0) & (arousal >= 0)))
q3_count = int(np.sum((valence < 0) & (arousal < 0)))
q4_count = int(np.sum((valence >= 0) & (arousal < 0)))
total_windows = len(valence)

print(f"=== Russell 2D Circumplex Distribution ({total_windows} windows, {duration_seconds:.1f}s) ===")
print(f"Quadrant I   [+V, +A] Exuberant  : {q1_count}/{total_windows} ({q1_count / total_windows * 100:.1f}%)")
print(f"Quadrant II  [-V, +A] Turbulent  : {q2_count}/{total_windows} ({q2_count / total_windows * 100:.1f}%)")
print(f"Quadrant III [-V, -A] Melancholic: {q3_count}/{total_windows} ({q3_count / total_windows * 100:.1f}%)")
print(f"Quadrant IV  [+V, -A] Calm       : {q4_count}/{total_windows} ({q4_count / total_windows * 100:.1f}%)")
dominant_q = max([("Exuberant", q1_count), ("Turbulent", q2_count), ("Melancholic", q3_count), ("Calm", q4_count)], key=lambda x: x[1])
print(f"Dominant Affective Persona: {dominant_q[0]} ({dominant_q[1] / total_windows * 100:.1f}% passage occupancy)")


## 8. Active Inquiry & Shannon Information-Gain Optimization
Select the most informative inquiry candidate window and generate Shannon information-maximizing questions with chip suggestions.

In [ ]:
active_engine = ActiveLearningEngine()
prompt_info = active_engine.formulate_inquiry_prompt(fused_analysis["segments"][0], [])

print("Suggested Active Inquiry Question:", prompt_info["question"])
print("Shannon Information-Gain Chips  :", prompt_info["chips"])

# Active Learning Simulation Benchmark vs Random Sampling
sim_result = active_engine.evaluate_ranking_simulation(fused_analysis["segments"])
print()
print(f"=== Active Inquiry Benchmark (Query Budget: {sim_result['query_budget']}) ===")
print(f"Active Query Accuracy: {sim_result['active_learning_accuracy'] * 100:.1f}%")
print(f"Random Prompt Accuracy: {sim_result['random_prompt_accuracy'] * 100:.1f}%")
print(f"Information Gain Delta: +{sim_result['improvement_delta'] * 100:.1f}%")


## 9. FAV Corpus (15s Favorite Passages) & Acoustic Contrast Corroboration
Classify natural-language listener explanations into the canonical 15-second FAV taxonomy and evaluate passage-vs-song acoustic contrast.

In [ ]:
# Example listener feedback
listener_comment = "Vocal intensity enters with deep emotion, the arrangement builds up and the rhythm drop is unforgettable."
classified_reasons = FavoritePassageTaxonomy().classify(listener_comment)
print("Listener Free-Text:", listener_comment)
print("Classified Canonical FAV Reasons:", classified_reasons)

# Dynamic passage selection adapted to the analyzed audio timeline
passage_start = start_offset
passage_end = min(start_offset + duration_seconds, passage_start + min(15.0, duration_seconds))
contrast = AcousticContrastAnalyzer().analyze_contrast(track_analysis, passage_start, passage_end)
print()
print(f"=== Acoustic Contrast Metrics ({passage_start:.1f}s - {passage_end:.1f}s) ===")
print(f"Energy Ratio (Passage / Track): {contrast.get('energy_ratio', 1.0):.2f}x")
print(f"Vocal Presence                : {contrast.get('vocal_presence', 0.0):.2f}")
print(f"Corroborated Musical Reasons  : {contrast.get('corroborated_reasons', [])}")


## 10. Complete Research Benchmark Suite
Execute the 6-pillar research integration benchmark suite across physical descriptors, neural representations, FAV methodology, active inquiries, and personalized emotion models.

In [ ]:
print("Executing full Research Benchmark Suite...")
benchmark_suite = BenchmarkSuite()
report = benchmark_suite.run()

print()
print("=================== BENCHMARK SUITE REPORT ===================")
print(f"Milestone: {report['milestone']}")
print(f"1. FAV Methodology Corroboration Rate : {report['1_fav_methodology']['acoustic_corroboration_rate'] * 100:.1f}%")
print(f"2. Deep Audio Encoder Latent Dim       : {report['2_deep_audio_encoder']['embedding_dim']}D ({report['2_deep_audio_encoder']['modelId']})")
print(f"3. DSP-Deep Fusion Dimensionality      : {report['3_dsp_deep_fusion']['fused_dim']}D (w_deep=0.7, w_dsp=0.3)")
print(f"4. Active Learning Accuracy Gain       : +{report['4_trompa_active_learning']['benchmark_vs_random']['improvement_delta'] * 100:.1f}%")
print(f"5. Personalized-DMER CCC Gain (Static) : +{report['5_personalized_dmer_baseline']['ccc_gain_over_static']:.4f}")
print(f"6. Concrete Illustrative Insight       : {report['concrete_illustrative_insight']['headline']}")
print("===============================================================")


## 11. Unified Time-Aligned Scientific Exploration Dashboard
Synchronizes all acoustic, neural, and affective representations along the identical horizontal timeline ($x \in [0, T]$ seconds with `sharex=True`).

### Vertically Aligned & Time-Synchronized Modalities:
1. **Time-Domain Audio Mixture Waveform** with dynamic RMS amplitude envelope
2. **High-Resolution Log-Frequency Spectrogram** ($N_{\text{FFT}}=2048$, $H=256$, $\approx 86$ fps, logarithmic frequency scale, $\text{dB}$ dynamic range)
3. **Meta Demucs Deep Neural 4-Stem Waveforms** (Vocals, Other/Harmonium, Drums/Tabla, Bass)
4. **Deterministic Physical Acoustic DSP Descriptors** (Normalized RMS Energy, Brightness, Onset Groove Activity, Vocal Presence Ratio, Spectral Flux)
5. **Continuous Dynamic Affect Trajectories (AAAI 2025)** (Time-varying Valence positivity and Arousal energy curves with zero-reference fill)
6. **Latent Manifold Coordinates** ($PC_1, PC_2, PC_3$ progression through the 89-D multimodal space)

*The spectrogram colorbar uses a non-intrusive dedicated inset axis (`inset_axes`), guaranteeing that all 6 panels share the exact same pixel-perfect plot width and horizontal bounding box with synchronized 5-second vertical guide markers.*


In [ ]:
# Super-Resolution Time-Frequency Representation
# Modes: "mel" (256 high-density perceptual mel bands, recommended) or "stft" (zero-padded N_FFT=4096 STFT)
spectrogram_mode = "mel"

hop_length = 128   # 172 frames / second (ultra-fine temporal resolution)
win_length = 1024  # 46 ms analysis window (sharp percussive transients without temporal blur)
n_fft = 2048 if spectrogram_mode == "mel" else 4096

if spectrogram_mode == "mel":
    mel_spec = librosa.feature.melspectrogram(
        y=audio_waveform, sr=sample_rate, n_fft=n_fft, win_length=win_length,
        hop_length=hop_length, n_mels=256, fmin=40, fmax=sample_rate / 2.0, power=2.0
    )
    spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    y_axis_type = "mel"
    freq_label = "Frequency (Hz, Mel)"
else:
    stft_raw = librosa.stft(audio_waveform, n_fft=n_fft, win_length=win_length, hop_length=hop_length, window="hann")
    spec_db = librosa.amplitude_to_db(np.abs(stft_raw), ref=np.max)
    y_axis_type = "log"
    freq_label = "Frequency (Hz, Log)"

spec_times = librosa.frames_to_time(np.arange(spec_db.shape[1]), sr=sample_rate, hop_length=hop_length) + start_offset

# Calculate high-resolution RMS envelope for mixture waveform
hop_rms = 512
rms_envelope = librosa.feature.rms(y=audio_waveform, frame_length=2048, hop_length=hop_rms)[0]
rms_time = librosa.frames_to_time(np.arange(len(rms_envelope)), sr=sample_rate, hop_length=hop_rms) + start_offset

# Normalize physical DSP descriptors to [0, 1] for visual alignment on shared vertical scale
def normalize_to_unit_span(values: np.ndarray) -> np.ndarray:
    span = np.ptp(values)
    return (values - np.min(values)) / span if span > 1e-6 else np.zeros_like(values)

norm_energies = normalize_to_unit_span(energies)
norm_centroids = normalize_to_unit_span(centroids)
norm_onsets = normalize_to_unit_span(onsets)
norm_fluxes = normalize_to_unit_span([s["descriptors"]["spectral_flux"] for s in segments])
norm_vocals = np.array(vocals)

# Construct unified 6-panel figure with strictly synchronized time axis
fig, axes = plt.subplots(
    6, 1, figsize=(15, 18), sharex=True,
    gridspec_kw={"height_ratios": [1.1, 2.3, 1.8, 1.3, 1.2, 1.2]}
)

# Panel 1: Time-Domain Audio Mixture & RMS Envelope
axes[0].plot(time_axis, audio_waveform, color="#8EC5B8", alpha=0.85, linewidth=0.7, label="Audio Mixture Waveform")
axes[0].plot(rms_time, rms_envelope, color="#E06C75", linewidth=1.6, label="RMS Energy Envelope")
axes[0].plot(rms_time, -rms_envelope, color="#E06C75", linewidth=1.6)
axes[0].set_ylabel("Amplitude", fontsize=9, fontweight="bold")
axes[0].set_title(f"Unified Synchronized Acoustic Dashboard — {demo_audio_path.name}", fontsize=13, fontweight="bold", pad=12)
axes[0].legend(loc="upper right", framealpha=0.85, fontsize=8)

# Panel 2: Super-Resolution Spectrogram (High Dynamic Range -52 dB, 172 fps)
spec_img = librosa.display.specshow(
    spec_db, sr=sample_rate, hop_length=hop_length, x_axis="time", x_coords=spec_times,
    y_axis=y_axis_type, fmin=40, fmax=sample_rate / 2.0, cmap="magma", ax=axes[1],
    vmin=-52, vmax=0
)
axes[1].set_ylabel(freq_label, fontsize=9, fontweight="bold")

# DEDICATED INSET AXIS FOR COLORBAR: Prevents matplotlib from narrowing or shifting axes[1]!
cax = axes[1].inset_axes([1.012, 0.0, 0.016, 1.0])
cbar = fig.colorbar(spec_img, cax=cax, format="%+2.0f dB")
cbar.set_label("Power (dB)", fontsize=8)

# Panel 3: Demucs Stem Waveforms (Dynamic Multi-Stem)
stem_colors = {"vocals": "#E06C75", "other": "#61AFEF", "drums": "#98C379", "bass": "#D19A66", "guitar": "#E5C07B", "piano": "#C678DD"}
for name in demucs_stems.keys():
    axes[2].plot(time_axis, demucs_stems[name], label=f"{name.capitalize()} Stem", color=stem_colors.get(name, "#ABB2BF"), alpha=0.75, linewidth=0.6)
axes[2].set_ylabel("Stem Amplitudes", fontsize=9, fontweight="bold")
axes[2].legend(loc="upper right", ncol=min(6, len(demucs_stems)), framealpha=0.85, fontsize=8)
# Panel 4: Deterministic Physical Acoustic Descriptors
axes[3].plot(midpoints, norm_energies, marker="o", markersize=5, color="#E06C75", label="RMS Energy (norm)", linewidth=1.8)
axes[3].plot(midpoints, norm_centroids, marker="s", markersize=5, color="#D19A66", label="Spectral Brightness (norm)", linewidth=1.8)
axes[3].plot(midpoints, norm_onsets, marker="^", markersize=5, color="#98C379", label="Onset Groove Activity (norm)", linewidth=1.8)
axes[3].plot(midpoints, norm_vocals, marker="d", markersize=5, color="#61AFEF", label="Vocal Presence Ratio", linewidth=1.8)
axes[3].plot(midpoints, norm_fluxes, marker="x", markersize=5, color="#C678DD", label="Spectral Flux (norm)", linewidth=1.8)
axes[3].set_ylabel("Normalized DSP", fontsize=9, fontweight="bold")
axes[3].set_ylim(-0.05, 1.15)
axes[3].legend(loc="upper right", ncol=5, framealpha=0.85, fontsize=8)

# Panel 5: Dynamic Affect Trajectories (AAAI 2025 Dual-Scale Temporal Attention)
axes[4].plot(midpoints, valence, marker="s", markersize=5, color="#98C379", label="Dynamic Valence (Positivity)", linewidth=2.0)
axes[4].plot(midpoints, arousal, marker="^", markersize=5, color="#E06C75", label="Dynamic Arousal (Energy / Tension)", linewidth=2.0)
axes[4].axhline(0.0, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)
axes[4].fill_between(midpoints, 0, valence, where=(valence >= 0), color="#98C379", alpha=0.15)
axes[4].fill_between(midpoints, 0, valence, where=(valence < 0), color="#98C379", alpha=0.08)
axes[4].fill_between(midpoints, 0, arousal, where=(arousal >= 0), color="#E06C75", alpha=0.15)
axes[4].fill_between(midpoints, 0, arousal, where=(arousal < 0), color="#E06C75", alpha=0.08)
axes[4].set_ylabel("Affect [-1, +1]", fontsize=9, fontweight="bold")
axes[4].set_ylim(-1.1, 1.1)
axes[4].legend(loc="upper right", ncol=2, framealpha=0.85, fontsize=8)

# Panel 6: Latent Manifold Coordinates over Time
axes[5].plot(midpoints, coords_3d[:, 0], marker="o", markersize=5, color="#61AFEF", label="Latent PC 1 (Macro Timbre/Structure)", linewidth=1.8)
axes[5].plot(midpoints, coords_3d[:, 1], marker="s", markersize=5, color="#E5C07B", label="Latent PC 2 (Harmonic Progression)", linewidth=1.8)
axes[5].plot(midpoints, coords_3d[:, 2], marker="^", markersize=5, color="#C678DD", label="Latent PC 3 (Dynamic Expression)", linewidth=1.8)
axes[5].set_ylabel("Latent Coordinates", fontsize=9, fontweight="bold")
axes[5].set_xlabel("Song Timeline (seconds)", fontsize=10, fontweight="bold")
axes[5].legend(loc="upper right", ncol=3, framealpha=0.85, fontsize=8)

# Dynamic timeline grid marks that adapt to duration
start_t = float(start_offset)
end_t = float(start_offset + duration_seconds)
step_s = max(2.0, min(30.0, round(duration_seconds / 12.0 / 5.0) * 5.0))
sync_ticks = np.arange(start_t, end_t + 0.1, step_s)

for i, ax in enumerate(axes):
    ax.set_xlim(start_t, end_t)
    ax.set_xticks(sync_ticks)
    ax.grid(True, which="major", axis="x", color="#4B5263", linestyle="--", linewidth=0.8, alpha=0.55)
    ax.grid(True, which="major", axis="y", color="#ABB2BF", linestyle=":", linewidth=0.5, alpha=0.35)

# Align Y-labels horizontally across all panels
fig.align_ylabels(axes)

# Apply tight_layout with reserved margin for the right colorbar
plt.tight_layout(rect=[0.01, 0.01, 0.93, 0.98])

output_img_path = workspace_root / "research-output" / "time_aligned_dashboard.png"
output_img_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(str(output_img_path), dpi=200)
print(f"✓ Saved super-resolution synchronized dashboard (200 DPI) to: {output_img_path}")
plt.show()
